# GA Optimized EMD-LSTM-XGBoost Commodity Forecasting
## Notebook 1: Data Preparation, Stationarity Testing and EMD Decomposition

Implementation follows the methodology described in the paper: preprocessing, Min-Max normalization, temporal split, ADF/KPSS testing and EMD decomposition.

In [ ]:
# Install required packages if needed
# !pip install numpy pandas matplotlib scikit-learn PyEMD statsmodels tensorflow xgboost deap


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.preprocessing import MinMaxScaler
from PyEMD import EMD
from statsmodels.tsa.stattools import adfuller, kpss


In [ ]:
# Load commodity price data
# CSV format: Date, Price
data = pd.read_csv('commodity_price.csv')
data['Date'] = pd.to_datetime(data['Date'])
data = data.sort_values('Date')
data['Price'] = data['Price'].interpolate()
price = data['Price'].values.reshape(-1,1)
data.head()


In [ ]:
# 80:20 chronological train test split
split = int(len(price)*0.8)
train = price[:split]
test = price[split:]

# Normalization fitted only on training data
scaler = MinMaxScaler()
train_scaled = scaler.fit_transform(train)
test_scaled = scaler.transform(test)


In [ ]:
# Stationarity tests
def stationarity_test(series):
    print('ADF p-value:', adfuller(series)[1])
    print('KPSS p-value:', kpss(series, regression='c')[1])

stationarity_test(train_scaled.flatten())


In [ ]:
# Empirical Mode Decomposition
emd = EMD()
imfs = emd(train_scaled.flatten())
print('Number of IMFs:', imfs.shape[0])

plt.figure(figsize=(12,8))
for i, imf in enumerate(imfs):
    plt.subplot(imfs.shape[0],1,i+1)
    plt.plot(imf)
    plt.title(f'IMF {i+1}')
plt.tight_layout()
